# AIC Smoke Test

This notebook runs the smoke gate for the AIC training pipeline.

**Hard requirements:**
- After SFT: parse rate >= 70% on 8 sampled prompts.
- During GRPO: at least one step has `reward_std > 0`.

If either gate fails, do **not** run `01_full_pipeline.ipynb` until the
issue is fixed. Inspect `logs/pipeline/stage_smoke.json` for diagnostics.

Total wall-clock target: ~8 minutes on T4 medium.

In [ ]:
import os, sys
REPO = '/workspace/aic-repo'
os.chdir(REPO)
sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

In [ ]:
!nvidia-smi || echo '(no GPU visible)'

In [ ]:
!python scripts/run_pipeline.py verify

In [ ]:
import subprocess
result = subprocess.run(['python', 'scripts/run_pipeline.py', 'smoke'], capture_output=False)
print('exit code:', result.returncode)

In [ ]:
import json
from pathlib import Path
log = Path('logs/pipeline/stage_smoke.json')
if log.exists():
    payload = json.loads(log.read_text())
    print(json.dumps({
        'status': payload.get('status'),
        'parse_rate_gate_passed': payload.get('parse_rate_gate_passed'),
        'reward_std_gate_passed': payload.get('reward_std_gate_passed'),
        'parse_rate': payload.get('sft', {}).get('parse_rate'),
        'max_reward_std': payload.get('grpo', {}).get('max_reward_std'),
    }, indent=2))
else:
    print('no smoke log found')

In [ ]:
from pathlib import Path
log = Path('logs/grpo_progress.jsonl')
if log.exists():
    print(log.read_text())
else:
    print('no grpo progress log yet')